# Pipeline de Alfabetização no Brasil — Notebook 4: Streaming Simulado

**Tech Challenge Fase 2 — FIAP POSTECH**

---

## Objetivo

Simular a ingestão **Streaming** de eventos em tempo quase real:

- Atualizações de indicadores de alfabetização (ex.: avaliações intermediárias)
- Revisões de metas municipais
- Novas medições de desempenho

## Por que Streaming?

O **Batch** cobre dados históricos completos (censos, avaliações anuais).  
O **Streaming** complementa com eventos que ocorrem entre as janelas batch:

```
Batch (anual) ──► dados consolidados
Streaming     ──► alertas, atualizações em tempo quase real
```

## Arquitetura Streaming na AWS

```
Gerador de Eventos (Python)  →  AWS Kinesis Data Streams
                                       ↓
                             AWS Lambda / Glue Streaming
                                       ↓
                             S3 Bronze/streaming/
                                       ↓
                             Micro-batch → Silver → Gold
```

> **Neste notebook:** simulamos o fluxo completo localmente com um arquivo de fila JSONL,
> demonstrando a mesma lógica que seria aplicada com Kinesis em produção.

## 1. Imports e Configuração

In [ ]:
import os
import io
import json
import time
import random
import logging
import threading
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import boto3

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"))
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

S3_BUCKET      = os.getenv("S3_BUCKET_NAME", "tech-challenge-alfabetizacao-01")
AWS_REGION     = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
USE_AWS        = os.getenv("USE_AWS", "false").lower() == "true"
QUEUE_FILE     = Path("/tmp/alfabetizacao_streaming_queue.jsonl")
LOCAL_STREAM   = Path("../data/bronze/streaming")
LOCAL_STREAM.mkdir(parents=True, exist_ok=True)

print(f"Modo: {'AWS Kinesis' if USE_AWS else 'LOCAL (fila JSONL)'}")
print(f"Fila: {QUEUE_FILE}")

## 2. Gerador de Eventos (Producer)

Simula eventos que chegariam de sistemas externos: avaliações municipais, atualizações de secretarias.

In [ ]:
# Dados reais dos datasets INEP para gerar eventos realistas
SIGLAS_UF = [
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA",
    "MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN",
    "RS","RO","RR","SC","SP","SE","TO"
]

# Códigos de município reais (primeiros 2 dígitos = código UF)
ID_MUNICIPIOS_AMOSTRA = [
    "3550308",  # São Paulo
    "3304557",  # Rio de Janeiro
    "5300108",  # Brasília
    "2304400",  # Fortaleza
    "2927408",  # Salvador
    "4106902",  # Curitiba
    "3106200",  # Belo Horizonte
    "1302603",  # Manaus
    "1501402",  # Belém
    "4314902",  # Porto Alegre
]

EVENT_TYPES = [
    "indicador_atualizado",
    "meta_revisada",
    "medicao_desempenho",
    "avaliacao_municipal",
]

REDES = ["Municipal", "Estadual"]


def generate_event(seq: int) -> dict:
    """Gera um evento de atualização de indicador."""
    sigla_uf = random.choice(SIGLAS_UF)
    return {
        "event_id"   : f"evt_{seq:05d}_{int(time.time() * 1000) % 100000}",
        "event_type" : random.choice(EVENT_TYPES),
        "timestamp"  : datetime.now(timezone.utc).isoformat(),
        "source"     : "streaming_simulado_notebook",
        "payload": {
            "sigla_uf"          : sigla_uf,
            "id_municipio"      : random.choice(ID_MUNICIPIOS_AMOSTRA),
            "ano"               : random.choice([2023, 2024, 2025]),
            "serie"             : 2,  # 2º ano EF (foco do INEP)
            "rede"              : random.choice(REDES),
            "taxa_alfabetizacao": round(random.uniform(40.0, 95.0), 2),
            "media_portugues"   : round(random.uniform(700.0, 800.0), 2),
            "fonte"             : "secretaria_educacao",
        }
    }


# Demonstração
sample_event = generate_event(1)
print("Exemplo de evento gerado:")
print(json.dumps(sample_event, indent=2, ensure_ascii=False))

## 3. Publicação de Eventos (Producer Run)

In [ ]:
def run_producer(n_events: int = 50, interval_ms: int = 100):
    """
    Publica N eventos na fila JSONL.
    Em produção: substituir por boto3.client('kinesis').put_record()
    """
    logger.info("Producer iniciado: %d eventos", n_events)
    QUEUE_FILE.parent.mkdir(parents=True, exist_ok=True)

    published = []
    with open(QUEUE_FILE, "w") as f:  # 'w' = nova fila
        for i in range(1, n_events + 1):
            event = generate_event(i)
            f.write(json.dumps(event) + "\n")
            f.flush()
            published.append(event)
            if interval_ms > 0:
                time.sleep(interval_ms / 1000)

    logger.info("Producer encerrado: %d eventos publicados em %s", n_events, QUEUE_FILE)
    return published


events = run_producer(n_events=50, interval_ms=0)  # sem delay em notebook
print(f"\n{len(events)} eventos publicados.")
print(f"Tipos: {pd.Series([e['event_type'] for e in events]).value_counts().to_dict()}")

## 4. Consumidor de Eventos (Consumer)

Lê a fila, agrupa em micro-batches e persiste no Bronze/streaming.

In [ ]:
FLUSH_EVERY = 10  # micro-batch: persiste a cada 10 eventos


def parse_events(events: list) -> pd.DataFrame:
    """Converte lista de eventos em DataFrame."""
    rows = []
    for e in events:
        row = e["payload"].copy()
        row["event_id"]        = e["event_id"]
        row["event_type"]      = e["event_type"]
        row["event_timestamp"] = e["timestamp"]
        row["source"]          = e.get("source", "streaming")
        rows.append(row)
    return pd.DataFrame(rows)


def save_micro_batch(df: pd.DataFrame, batch_num: int):
    """Persiste micro-batch como Parquet no Bronze/streaming."""
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    ts    = datetime.now(timezone.utc).strftime("%H%M%S")

    if USE_AWS:
        key = f"bronze/streaming/dt={today}/batch_{ts}_{batch_num:03d}.parquet"
        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False, engine="pyarrow")
        buffer.seek(0)
        boto3.client("s3", region_name=AWS_REGION).put_object(
            Bucket=S3_BUCKET, Key=key, Body=buffer.getvalue()
        )
        logger.info("S3 batch %03d: s3://%s/%s (%d eventos)", batch_num, S3_BUCKET, key, len(df))
    else:
        local_path = LOCAL_STREAM / f"dt={today}"
        local_path.mkdir(parents=True, exist_ok=True)
        out = local_path / f"batch_{ts}_{batch_num:03d}.parquet"
        df.to_parquet(out, index=False, engine="pyarrow")
        logger.info("Local batch %03d: %s (%d eventos)", batch_num, out, len(df))


def run_consumer():
    """Lê a fila JSONL e processa em micro-batches."""
    if not QUEUE_FILE.exists():
        logger.error("Fila não encontrada: %s. Execute o producer primeiro.", QUEUE_FILE)
        return pd.DataFrame()

    all_events = []
    with open(QUEUE_FILE, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                all_events.append(json.loads(line))
            except json.JSONDecodeError:
                logger.warning("Linha inválida ignorada: %s", line[:80])

    logger.info("Consumer leu %d eventos da fila.", len(all_events))

    # Processa em micro-batches
    batch_dfs = []
    for i in range(0, len(all_events), FLUSH_EVERY):
        batch = all_events[i:i + FLUSH_EVERY]
        df_batch = parse_events(batch)
        save_micro_batch(df_batch, batch_num=i // FLUSH_EVERY + 1)
        batch_dfs.append(df_batch)

    return pd.concat(batch_dfs, ignore_index=True) if batch_dfs else pd.DataFrame()


streaming_df = run_consumer()
print(f"\nConsumo concluído: {len(streaming_df)} eventos processados")
display(streaming_df.head(5))

## 5. Merge Streaming → Bronze Consolidado

In [ ]:
print("Análise dos eventos de streaming:")
print("-" * 40)
print(f"Total de eventos    : {len(streaming_df):,}")
print(f"UFs cobertas        : {streaming_df['sigla_uf'].nunique()}")
print(f"Tipos de evento:")
print(streaming_df["event_type"].value_counts().to_string())
print()
print("Estatísticas da taxa de alfabetização nos eventos:")
print(streaming_df["taxa_alfabetizacao"].describe().round(2).to_string())

In [ ]:
# Valida consistência dos eventos
issues = []
n_nulls = streaming_df[["sigla_uf", "id_municipio", "taxa_alfabetizacao"]].isnull().sum()
n_dupes = streaming_df.duplicated(subset=["event_id"]).sum()
out_of_range = (~streaming_df["taxa_alfabetizacao"].between(0, 100)).sum()

print("Validação de qualidade dos eventos de streaming:")
print(f"  Nulos em campos-chave : {n_nulls.sum()}")
print(f"  Event IDs duplicados  : {n_dupes}")
print(f"  Taxa fora de 0-100    : {out_of_range}")

if n_nulls.sum() == 0 and n_dupes == 0 and out_of_range == 0:
    print("\n[OK] Todos os eventos são válidos.")
else:
    print("\n[AVISO] Problemas detectados — veja detalhes acima.")

## 6. Como Usar Kinesis em Produção

O código abaixo mostra como substituir a fila JSONL por **AWS Kinesis Data Streams**:

In [ ]:
# REFERÊNCIA — não executar sem credenciais AWS e stream criado

kinesis_producer_example = '''
import boto3, json

kinesis = boto3.client("kinesis", region_name="us-east-1")
STREAM_NAME = "alfabetizacao-indicadores"

def publish_to_kinesis(event: dict):
    kinesis.put_record(
        StreamName=STREAM_NAME,
        Data=json.dumps(event).encode(),
        PartitionKey=event["payload"]["sigla_uf"]  # partição por UF
    )
'''

kinesis_consumer_example = '''
import boto3, json

kinesis = boto3.client("kinesis", region_name="us-east-1")
STREAM_NAME = "alfabetizacao-indicadores"

def consume_kinesis(shard_iterator):
    response = kinesis.get_records(ShardIterator=shard_iterator, Limit=100)
    events = [json.loads(r["Data"]) for r in response["Records"]]
    # processar e salvar no S3 Bronze/streaming/
    return events, response.get("NextShardIterator")
'''

print("Producer Kinesis:")
print(kinesis_producer_example)
print("Consumer Kinesis:")
print(kinesis_consumer_example)

---
## Decisões Arquiteturais — Streaming

| Decisão | Escolha | Alternativa | Motivo |
|---|---|---|---|
| Simulação local | Arquivo JSONL | Kafka, RabbitMQ | Evita custo de infraestrutura em dev/teste |
| AWS Kinesis (produção) | Kinesis Data Streams | Apache Kafka | Gerenciado pela AWS, integrado ao ecossistema |
| Micro-batch de 10 eventos | Flush a cada 10 | Lambda por evento | Reduz custo S3 PUT; balanceia latência vs custo |
| Parquet no Bronze streaming | Parquet | JSON raw | Consistência de formato com o pipeline Batch |
| Partição por `dt=YYYY-MM-DD` | Hive-style | Por hora | Dados educacionais não exigem sub-hora |

**Próximo passo:** execute `05_quality_checks.ipynb`